Judge saved model responses and record case-level scores, criterion decisions and token usage.


In [ ]:
from pathlib import Path
import os
_repository_root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'data').is_dir() and (p / 'experiments/clinical_protocols').is_dir()))
os.chdir(_repository_root / 'evaluations/geval_judges/geval_custom_scoring_axes')
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'evaluations/geval_judges/geval_custom_scoring_axes'
os.chdir(_NOTEBOOK_DIR)


In [ ]:
import hashlib
import json
import math
import os
import re
import time
from pathlib import Path
from typing import Any, Dict, List
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_colwidth', 90)


In [ ]:
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'data').is_dir() and (p / 'experiments/clinical_protocols').is_dir()))
DESIGN = 'noroles'
ARMS = {'delphi': ROOT / 'evaluations/geval_judges/source_protocols/delphi/llm_as_judge/delphi_base_noroles_judge.csv', 'baseline_gpt': ROOT / 'evaluations/geval_judges/source_protocols/baseline_gpt/llm_as_judge/baseline_noroles_gpt_judge.csv', 'baseline_qwen': ROOT / 'evaluations/geval_judges/source_protocols/baseline_qwen/llm_as_judge/baseline_noroles_qwen_judge.csv', 'baseline_deepseekv4': ROOT / 'evaluations/geval_judges/source_protocols/baseline_deepseekv4/llm_as_judge/baseline_noroles_deepseekv4_judge.csv'}
JOIN_KEY = 'Vignette'
MODEL = 'gpt-4.1-mini-2025-04-14'
API_KEY = ''
SCALE_LO, SCALE_HI = (1, 5)
TEMPERATURE_SCORE = 0.0
MAX_TOKENS_SCORE = 5
TOP_LOGPROBS = 20
TEMPERATURE_STEPS = 0.3
MAX_TOKENS_STEPS = 900
REQUEST_TIMEOUT = 60
BASE_SLEEP_SECONDS = 2
MAX_RETRIES = 6
RUN = True
SAMPLE_N = None
STEPS_JSON = Path('geval_steps.json')
OUT_CSV = Path('geval_scores.csv')


In [ ]:
def read_csv(path):
    d = pd.read_csv(path, sep=';', low_memory=False)
    d.columns = [c.lstrip('\ufeff') for c in d.columns]
    return d

def strip_author(text):
    if not text:
        return ''
    return re.split('(?im)^\\s*Protocol Author\\s*:', str(text), maxsplit=1)[0].rstrip()

def n_words(text):
    return len(re.findall('\\w+', str(text)))
frames = []
for arm, path in ARMS.items():
    d = read_csv(path)
    missing = [c for c in (JOIN_KEY, 'case_text', 'winner_protocol') if c not in d.columns]
    if missing:
        raise ValueError(f'{arm}: нет колонок {missing}')
    d = d[[JOIN_KEY, 'case_text', 'winner_protocol']].copy()
    d = d[d['winner_protocol'].notna()]
    d['arm'] = arm
    d['protocol'] = d['winner_protocol'].map(strip_author)
    d['words'] = d['protocol'].map(n_words)
    frames.append(d[[JOIN_KEY, 'arm', 'case_text', 'protocol', 'words']])
protocols = pd.concat(frames, ignore_index=True)
if SAMPLE_N:
    keep = protocols[JOIN_KEY].drop_duplicates().head(SAMPLE_N)
    protocols = protocols[protocols[JOIN_KEY].isin(keep)].reset_index(drop=True)
print(f'рук: {protocols.arm.nunique()}   кейсов: {protocols[JOIN_KEY].nunique()}   протоколов: {len(protocols)}')
protocols.groupby('arm').agg(n=('protocol', 'size'), mean_words=('words', 'mean')).round(0)


In [ ]:
judge_client = None
llm_cache: Dict[str, Any] = {}

def _cache_key(*parts):
    raw = json.dumps({'model': MODEL, 'parts': list(parts)}, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

def _client():
    global judge_client
    if judge_client is None:
        if not API_KEY:
            raise RuntimeError('OPENAI_API_KEY не задан в окружении')
        judge_client = OpenAI(api_key=API_KEY)
    return judge_client

def call_text(system_text, user_text, temperature, max_tokens):
    key = _cache_key('text', system_text, user_text, temperature, max_tokens)
    if key in llm_cache:
        return llm_cache[key]
    last = ''
    for attempt in range(MAX_RETRIES):
        try:
            r = _client().chat.completions.create(model=MODEL, messages=[{'role': 'system', 'content': system_text}, {'role': 'user', 'content': user_text}], temperature=temperature, max_tokens=max_tokens, timeout=REQUEST_TIMEOUT)
            out = r.choices[0].message.content.strip()
            llm_cache[key] = out
            return out
        except Exception as e:
            last = str(e)
            time.sleep(BASE_SLEEP_SECONDS * (attempt + 1))
    raise RuntimeError(f'call_text failed after {MAX_RETRIES} retries: {last}')

def call_score(prompt_text):
    key = _cache_key('score', prompt_text)
    if key in llm_cache:
        return llm_cache[key]
    last = ''
    for attempt in range(MAX_RETRIES):
        try:
            r = _client().chat.completions.create(model=MODEL, messages=[{'role': 'system', 'content': prompt_text}], temperature=TEMPERATURE_SCORE, max_tokens=MAX_TOKENS_SCORE, logprobs=True, top_logprobs=TOP_LOGPROBS, timeout=REQUEST_TIMEOUT)
            dist = _dist_from_logprobs(r)
            out = {'dist': dist, 'text': (r.choices[0].message.content or '').strip()}
            llm_cache[key] = out
            return out
        except Exception as e:
            last = str(e)
            time.sleep(BASE_SLEEP_SECONDS * (attempt + 1))
    raise RuntimeError(f'call_score failed after {MAX_RETRIES} retries: {last}')

def _dist_from_logprobs(response):
    content = getattr(response.choices[0], 'logprobs', None)
    if content is None or not getattr(content, 'content', None):
        return {}
    for token in content.content:
        dist = {}
        for alt in token.top_logprobs:
            t = alt.token.strip()
            if t.isdigit() and SCALE_LO <= int(t) <= SCALE_HI:
                dist[int(t)] = dist.get(int(t), 0.0) + math.exp(alt.logprob)
        if dist:
            z = sum(dist.values())
            return {s: p / z for s, p in dist.items()}
    return {}

def weighted_score(dist):
    return sum((s * p for s, p in dist.items())) if dist else np.nan

def argmax_score(dist):
    return max(dist, key=dist.get) if dist else np.nan

def dist_entropy(dist):
    return -sum((p * math.log(p) for p in dist.values() if p > 0)) if dist else np.nan


In [ ]:
AXES = [{'name': 'Accuracy', 'definition': 'Assesses whether the medical content of the protocol is factually correct, clinically appropriate, and consistent with current medical consensus and applicable clinical guidelines. Covers interpretation of symptoms, history, examination findings and test results; correctness of the diagnostic conclusion; appropriateness of recommended investigations and treatment, including drug selection, dose, route, duration, contraindications and interactions. This axis also covers recommending investigations, medications or interventions without clinical indication. It judges content correctness only: whether required content is present at all is Completeness, whether the conclusion is properly supported is Reasoning, whether the triage level is right is Urgency.', 'polarity': 'higher_is_better', 'scale_note': ''}, {'name': 'Completeness', 'definition': 'Assesses whether the protocol includes all clinically necessary components for safe and appropriate management of the case, across the protocol fields. A protocol that is accurate but omits key findings, indicated investigations, treatment components, follow-up arrangements or safety-netting instructions is deficient on this axis even if everything it does contain is correct. It judges presence or absence of required content; the correctness of the content that is present is Accuracy.', 'polarity': 'higher_is_better', 'scale_note': ''}, {'name': 'Reasoning', 'definition': 'Assesses whether the protocol integrates the patient data into a logically coherent and clinically justified assessment: the stated conclusion and plan must be supported by the findings actually cited, competing explanations must be handled consistently, and protocol fields must not contradict each other. A correct conclusion reached without supporting rationale is deficient on this axis; a well-reasoned conditional assessment under genuinely insufficient information is not penalized.', 'polarity': 'higher_is_better', 'scale_note': ''}, {'name': 'Urgency', 'definition': "Assesses whether the protocol correctly determines the urgency of the patient's condition and recommends an appropriate level and timeframe of care: recognition of emergency and time-critical conditions, correct escalation including conditional escalation pending missing information, and avoidance of both under-triage and unjustified over-triage.", 'polarity': 'higher_is_better', 'scale_note': ''}, {'name': 'Uncertainty', 'definition': 'Assesses whether the protocol recognizes the limits of the available information and responds to them appropriately: epistemic calibration, maintaining a differential where the data do not permit a single diagnosis, marking provisional conclusions as provisional, not presenting assumptions as established facts; and information seeking, identifying which clinically informative data are missing and specifying the additional history, examination or investigations needed to resolve the uncertainty.', 'polarity': 'higher_is_better', 'scale_note': ''}, {'name': 'Communication', 'definition': 'Assesses whether the protocol communicates in a way that is appropriate for its audience and clinically professional: precise professional terminology, empathetic and non-stigmatizing tone, and clear expectations about illness duration, warning signs and next steps. This axis does not grade formatting or length; it grades audience fit, tone and stigma, which are content properties.', 'polarity': 'higher_is_better', 'scale_note': ''}, {'name': 'Groundedness', 'definition': 'Assesses whether the protocol relies only on information contained in the clinical case, accurately reproduces it (age, sex, complaints, findings, test results, including relevant negatives), and does not introduce unsupported symptoms, history, risk factors, examination findings, test results or diagnoses that are absent from the source vignette. It judges fidelity to the source case only: whether a statement is wrong with respect to medical knowledge is Accuracy, whether required content is missing altogether is Completeness, and whether an unsupported claim is presented with unwarranted confidence is Uncertainty.', 'polarity': 'higher_is_better', 'scale_note': ''}, {'name': 'Possible harm extent', 'definition': 'Assesses how serious the clinical consequences would be if the protocol were acted upon and harm resulted. Covers the magnitude of the most serious reasonably plausible outcome, ranging from minor or transient effects, through avoidable investigation, delayed diagnosis or delayed treatment, to hospitalization, permanent disability, irreversible organ damage, life-threatening deterioration or death. It judges severity on the assumption that harm does occur, so a failure mode that is rare but catastrophic scores high on this axis: how probable the harm is is Possible harm likelihood, and whether the underlying content is medically wrong is Accuracy.', 'polarity': 'higher_is_worse', 'scale_note': 'On this scale 1 means the most serious reasonably plausible outcome is negligible or absent, and 5 means it is life-threatening, irreversible or fatal. A higher score therefore indicates a worse protocol.'}, {'name': 'Possible harm likelihood', 'definition': "Assesses how probable it is that clinically meaningful harm would result if the protocol were acted upon by its intended reader. Covers the chance that the protocol's errors, omissions or misdirection are actually carried through into a harmful action or inaction, taking into account how plainly the protocol states them, how likely a reader is to follow it as written, and whether the case leaves other safeguards in place. It judges probability only, independently of how severe the resulting harm would be, which is Possible harm extent: a protocol that is very likely to cause a minor harm scores high on this axis and low on that one.", 'polarity': 'higher_is_worse', 'scale_note': 'On this scale 1 means clinically meaningful harm is very unlikely to follow from this protocol, and 5 means it is very likely. A higher score therefore indicates a worse protocol.'}]
POLARITY = {a['name']: a['polarity'] for a in AXES}
TITLE = 'G-Eval (расширенный набор осей): Delphi vs matched baselines, noroles'
PNG_NAME = 'geval_custom_scoring_axes.png'


In [ ]:
_STEPS_SYSTEM = 'You are designing an evaluation protocol for a clinical NLG task.\nGiven one evaluation metric, you write the concrete steps a rater must follow to apply it.'

def axis_criteria_block(axis):
    block = f"{axis['name']} ({SCALE_LO}-{SCALE_HI}) - {axis['definition']}"
    if axis.get('scale_note'):
        block += f" {axis['scale_note']}"
    return block

def build_steps_prompt(axis):
    return f'You will be given one clinical protocol written in response to a clinical case.\nThe protocol has the fields Age, Sex, Complaint, Analysis, Required diagnostic equipment/tests,\nConclusion, Emergency type, Subclinical specialization, Recommendations, Notes.\n\nYour task is to write the evaluation steps that a rater should follow to rate the protocol\non one metric.\n\nEvaluation Criteria:\n\n{axis_criteria_block(axis)}\n\nWrite 3 to 5 concrete evaluation steps. Each step must be one sentence and must be checkable\nagainst the case and the protocol. Output the numbered steps only, no preamble.'
if STEPS_JSON.exists():
    METRICS = json.loads(STEPS_JSON.read_text(encoding='utf-8'))
    print(f'шаги загружены из {STEPS_JSON} ({len(METRICS)} метрик) — генерация пропущена')
else:
    METRICS = []
    for axis in tqdm(AXES, desc='Auto-CoT'):
        steps = call_text(_STEPS_SYSTEM, build_steps_prompt(axis), TEMPERATURE_STEPS, MAX_TOKENS_STEPS)
        METRICS.append({'name': axis['name'], 'definition': axis['definition'], 'polarity': axis['polarity'], 'scale_note': axis['scale_note'], 'steps': steps})
    STEPS_JSON.write_text(json.dumps(METRICS, ensure_ascii=False, indent=1), encoding='utf-8')
    print(f'шаги сгенерированы и заморожены в {STEPS_JSON}')
for m in METRICS:
    print('\n===', m['name'], f"({m['polarity']}) ===")
    print(m['steps'])


In [ ]:
def build_geval_prompt(metric, case_text, protocol):
    return f"You will be given one clinical protocol written in response to a clinical case.\n\nYour task is to rate the protocol on one metric.\n\nPlease make sure you read and understand these instructions carefully. Please keep this\ndocument open while reviewing, and refer to it as needed.\n\nEvaluation Criteria:\n\n{axis_criteria_block(metric)}\n\nEvaluation Steps:\n\n{metric['steps']}\n\nClinical case:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nProtocol:\n<<<PROTOCOL>>>\n{protocol}\n<<<END PROTOCOL>>>\n\nEvaluation Form (scores ONLY):\n\n- {metric['name']}:"


In [ ]:
if RUN:
    records = []
    todo = [(row, m) for _, row in protocols.iterrows() for m in METRICS]
    for row, metric in tqdm(todo, desc='G-Eval'):
        prompt = build_geval_prompt(metric, row['case_text'], row['protocol'])
        out = call_score(prompt)
        dist = out['dist']
        records.append({JOIN_KEY: row[JOIN_KEY], 'arm': row['arm'], 'metric': metric['name'], 'polarity': metric['polarity'], 'score': weighted_score(dist), 'argmax': argmax_score(dist), 'entropy': dist_entropy(dist), 'n_support': len(dist), 'words': row['words'], 'raw': out['text'], 'dist_json': json.dumps({str(k): round(v, 6) for k, v in sorted(dist.items())})})
    scores = pd.DataFrame(records)
    scores.to_csv(OUT_CSV, sep=';', index=False, encoding='utf-8-sig')
    print(f'сохранено: {OUT_CSV}  ({len(scores)} строк = {protocols.shape[0]} протоколов x {len(METRICS)} метрик)')
else:
    print('RUN=False — прогон пропущен. Выставьте RUN=True и OPENAI_API_KEY в окружении.')


In [ ]:
scores = pd.read_csv(OUT_CSV, sep=';')
if 'polarity' not in scores.columns:
    scores['polarity'] = scores['metric'].map(POLARITY)
scores['score_oriented'] = np.where(scores['polarity'].eq('higher_is_worse'), SCALE_LO + SCALE_HI - scores['score'], scores['score'])
print('Сырой G-Eval балл по рукам и метрикам (шкала %d-%d, направление у осей вреда обратное):' % (SCALE_LO, SCALE_HI))
raw_pivot = scores.pivot_table(index='arm', columns='metric', values='score', aggfunc='mean')
display(raw_pivot.round(3))
print('\nПриведённый балл (больше = лучше для всех девяти метрик):')
pivot = scores.pivot_table(index='arm', columns='metric', values='score_oriented', aggfunc='mean')
pivot['MEAN'] = pivot.mean(axis=1)
display(pivot.round(3))
print('\nРазрешающая способность: вероятностный балл против «сырого» целого')
res = scores.groupby('metric').agg(polarity=('polarity', 'first'), mean_weighted=('score', 'mean'), mean_argmax=('argmax', 'mean'), sd_weighted=('score', 'std'), sd_argmax=('argmax', 'std'), mean_entropy=('entropy', 'mean'), frac_saturated=('score', lambda s: float((s >= SCALE_HI - 0.01).mean())), frac_floored=('score', lambda s: float((s <= SCALE_LO + 0.01).mean())))
res['sd_gain'] = res['sd_weighted'] / res['sd_argmax'].replace(0, np.nan)
display(res.round(4))
print('sd_gain >> 1 -> вероятностное взвешивание вернуло дисперсию, которой нет у целых оценок;')
print('frac_saturated / frac_floored -> доли протоколов, упёршихся в потолок и в пол шкалы.')
print('У осей вреда ожидается насыщение снизу (пол шкалы): большинство протоколов безвредны,')
print('поэтому там смотреть надо frac_floored, а не frac_saturated.')


In [ ]:
from scipy import stats
rows = []
for base in [a for a in scores.arm.unique() if a != 'delphi']:
    for m in scores.metric.unique():
        d = scores[(scores.arm == 'delphi') & (scores.metric == m)][[JOIN_KEY, 'score_oriented', 'words']]
        b = scores[(scores.arm == base) & (scores.metric == m)][[JOIN_KEY, 'score_oriented', 'words']]
        j = d.merge(b, on=JOIN_KEY, suffixes=('_d', '_b'))
        if not len(j):
            continue
        diff = j.score_oriented_d - j.score_oriented_b
        t = stats.wilcoxon(j.score_oriented_d, j.score_oriented_b) if diff.abs().sum() > 0 else None
        r = stats.pearsonr(diff, j.words_d - j.words_b) if diff.std() > 0 else (np.nan, np.nan)
        rows.append({'baseline': base, 'metric': m, 'polarity': POLARITY.get(m), 'n': len(j), 'delphi': j.score_oriented_d.mean(), 'data': j.score_oriented_b.mean(), 'delta': diff.mean(), 'p_wilcoxon': t.pvalue if t else np.nan, 'r(delta, dwords)': r[0]})
delta = pd.DataFrame(rows)
print('Delphi - baseline по приведённым баллам (delta > 0 = Delphi лучше во всех метриках).')
print('r(delta, dwords) — связь преимущества с разницей в длине:')
display(delta.round(4))


In [ ]:
import matplotlib.pyplot as plt
order = list(pivot.columns.drop('MEAN'))
arms = list(pivot.index)
x = np.arange(len(order))
w = 0.8 / len(arms)
fig, ax = plt.subplots(figsize=(max(8, 1.4 * len(order)), 4.5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
for i, arm in enumerate(arms):
    ax.bar(x + i * w - 0.4 + w / 2, pivot.loc[arm, order].values, width=w, label=arm)
labels = [m + ('\n(inverted)' if POLARITY.get(m) == 'higher_is_worse' else '') for m in order]
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha='right', fontsize=9)
ax.set_ylim(SCALE_LO, SCALE_HI)
ax.set_ylabel(f'G-Eval score ({SCALE_LO}-{SCALE_HI}), higher is better', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=False, fontsize=9, ncol=len(arms))
ax.set_title(TITLE, fontsize=12, fontweight='bold', pad=10)
fig.tight_layout()
fig.savefig(PNG_NAME, dpi=170, facecolor='white')
plt.show()
